# Proyecto
## **ING200** - OPTIMIZACIÓN
***Prof**. Jorge Acuña, Ph.D.*

**Integrantes:** *Simón Valdés, Vicente Díaz*

**Fecha:** *Ns*

*Universidad Adolfo Ibáñez*

*Viña del Mar, Chile*

In [1]:
import pandas as pd
from gurobipy import GRB
import gurobipy as gp
import matplotlib as plt

Para procesar el archivo de datos en Excel lo convertimos a dos archivos CSV, uno por cada tabla. Los cargamos como Data Frames a continuación.

In [2]:
operaciones = pd.read_csv("data/Operaciones.csv", sep=";")
permitidas = pd.read_csv("data/Operaciones_Permitidas.csv", sep=",")

In [3]:
# tratar datos de operaciones
#cambiamos coma por punto y pasamos a float
operaciones["Duracion_Horas"] = operaciones["Duracion_Horas"].str.replace(",", ".").astype(float)
#quitamos espacios y pasamos a entero
operaciones["Costo ($)"] = operaciones["Costo ($)"].str.replace(' ', '').astype(int)

operaciones.head()

,ID_Operacion,Duracion_Horas,Tipo,Costo ($)
0,OP001,3.5,Traumatológica,2910051
1,OP002,1.0,Neurológica,3270018
2,OP003,1.5,Gástrica,1840015
3,OP004,1.5,Traumatológica,2080043
4,OP005,3.5,Vascular,3800018


In [4]:
#tratar los datos de permitidas
#hay que convertir en una lista la columna de operaciones permitidas
permitidas['Tipos de operaciones permitidas'] = permitidas['Tipos de operaciones permitidas'].str.split(';')

permitidas.head()

,Pabellón,Tipos de operaciones permitidas
0,Pabellón_1,"[Dérmica, Ginecológica, Otorrinolaringológica,..."
1,Pabellón_2,"[Vascular, Urológica, Gástrica, Neurológica]"
2,Pabellón_3,"[Gástrica, Neurológica, Urológica, Oftalmológi..."
3,Pabellón_4,"[Otorrinolaringológica, Vascular, Ginecológica..."
4,Pabellón_5,"[Gástrica, Traumatológica, Oftalmológica]"


In [5]:
# queremos calcular las operaciones validas
# primero pasar a listas

dias = [1,2,3,4,5]
id_operaciones = operaciones["ID_Operacion"].tolist()
pabellones = permitidas["Pabellón"].tolist()

#hacemos diccionarios para cada operacion
duracion = dict(zip(operaciones["ID_Operacion"], operaciones["Duracion_Horas"]))
costo = dict(zip(operaciones["ID_Operacion"], operaciones["Costo ($)"]))
tipo = dict(zip(operaciones["ID_Operacion"], operaciones["Tipo"]))

#permitidas a diccionario
permitidas_dict = dict(zip(permitidas["Pabellón"], permitidas["Tipos de operaciones permitidas"]))

combinaciones = []
for i in id_operaciones:
    for j in dias:
        for k in pabellones:
            if tipo[i] in permitidas_dict[k]:
                combinaciones.append((i,j,k))

#print(combinaciones)
print(f"operaciones posibles = {len(dias)*len(id_operaciones) * len(pabellones)}")
print(f"operaciones validas = {len(combinaciones)}")

operaciones posibles = 2500
operaciones validas = 1105


In [6]:
# Modelo Base
model = gp.Model("Hospital_Zarcillo")

# Variables de Decisión
# X_i,j,k = 1 si la operacion 'i' se hace el día 'j' en el pabellón 'k', 0 si no.
# i = operacion, j = dia, k = pabellon
X = model.addVars(combinaciones, vtype=GRB.BINARY, name="Asignacion")

# Función Objetivo: Maximizar cantidad total de asignaciones
model.setObjective(X.sum(), GRB.MAXIMIZE)

# Restricciones

# 1) Presupuesto
# La suma de los costos de las operaciones seleccionadas no debe superar 100 millones
model.addConstr(
    gp.quicksum(X[i,j,k] * costo[i] for (i,j,k) in combinaciones) <= 100000000, 
    name="Presupuesto"
)

# 2) Asignación Única
# No repetir operaciones (cada operación se hace máximo 1 vez en toda la semana)
for i in id_operaciones:
    model.addConstr(
        gp.quicksum(X[i,j,k] for (op,j,k) in combinaciones if op == i) <= 1, 
        name=f"No_repeticion_{i}"
    )

# 3) Capacidad de Tiempo Diario
# No pasarse del tiempo permitido diario (10 hrs)
# Suma de (tiempo operacion + 1 hr limpieza) para todas las operaciones en un pabellon en un dia <= 11 hrs
for j in dias:
    for k in pabellones:
        model.addConstr(
            gp.quicksum(X[i,j,k] * (duracion[i] + 1) for (i,dia,pab) in combinaciones if dia == j and pab == k) <= 11,
            name=f"Tiempo_{j}_{k}"
        )

model.update()
print(f"Modelo creado. Variables: {model.NumVars}, Restricciones: {model.NumConstrs}")


Set parameter Username
Set parameter LicenseID to value 2831201
Academic license - for non-commercial use only - expires 2027-06-04
Modelo creado. Variables: 1105, Restricciones: 126


**Función Objetivo:**
$$\max Z = \sum_{i \in I} \sum_{j \in J} \sum_{k \in K} X_{i,j,k}$$

**s.a:**

1. **Presupuesto:**
$$\sum_{i \in I} \sum_{j \in J} \sum_{k \in K} X_{i,j,k} \cdot c_i \leq 100.000.000$$

2. **No se repiten las operaciones en otro día o pabellón:**
$$\sum_{j \in J} \sum_{k \in K} X_{i,j,k} \leq 1 \quad \forall i \in I$$

3. **Tiempo Diario Disponible:**
$$\sum_{i \in I} (t_i + 1) \cdot X_{i,j,k} \leq 11 \quad \forall j \in J, \forall k \in K$$

In [7]:
model.optimize()
print("Operaciones máximas a realizar:", model.ObjVal) #pregunta A

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 126 rows, 1105 columns and 3315 nonzeros (Max)
Model fingerprint: 0xb0d74440
Model has 1105 linear objective coefficients
Variable types: 0 continuous, 1105 integer (1105 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+06]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+08]

Found heuristic solution: objective 39.0000000
Presolve time: 0.01s
Presolved: 126 rows, 1105 columns, 3315 nonzeros
Variable types: 0 continuous, 1105 integer (1105 binary)

Root relaxation: objective 5.886840e+01, 284 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0   58.86840    0   11   39.00